In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor"
)

train_data = dataset["train"]

example = train_data[0]

print(example.keys())
print(example["question"])
print(example["answer"])
print(example["type"])
print(example["level"])
print(example["context"])


dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])
Which magazine was started first Arthur's Magazine or First for Women?
Arthur's Magazine
comparison
medium
{'title': ['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail', "Arthur's Magazine", '2014–15 Ukrainian Hockey Championship', 'First for Women', 'Freeway Complex Fire', 'William Rast'], 'sentences': [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008

In [5]:
def chunk_text(text, chunk_size=200, overlap=40):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [6]:
all_chunks = []

def extract_documents(example):
    documents = []

    titles = example["context"]["title"]
    sentences = example["context"]["sentences"]

    for title, article_sentences in zip(titles, sentences):

        text = " ".join(article_sentences)

        documents.append({
            "source": title,
            "text": text
        })

    return documents

documents = extract_documents(example)

# for document in documents:
#     print("=" * 80)
#     print("SOURCE:", document["source"])
#     print(document["text"][:500])


In [7]:
def chunk_sentences(sentences, chunk_size=5, overlap=1):
    chunks = []

    start = 0

    while start < len(sentences):

        end = start + chunk_size

        chunk = " ".join(sentences[start:end])

        if chunk:
            chunks.append({
                "text": chunk,
                "start_sentence": start,
                "end_sentence": min(end, len(sentences)) - 1
            })

        start += chunk_size - overlap

    return chunks

In [8]:
unique_chunks = {}

for example_id, example in enumerate(train_data):

    titles = example["context"]["title"]
    articles = example["context"]["sentences"]

    for title, sentences in zip(titles, articles):

        chunks = chunk_sentences(
            sentences,
            chunk_size=5,
            overlap=1
        )

        for chunk_id, chunk in enumerate(chunks):

            key = (
                title,
                chunk["text"]
            )

            if key not in unique_chunks:

                unique_chunks[key] = {
                    "text": chunk["text"],
                    "source": title,
                    "chunk_id": chunk_id,
                    "start_sentence": chunk["start_sentence"],
                    "end_sentence": chunk["end_sentence"]
                }

all_chunks = list(unique_chunks.values())

print("Unique chunks:", len(all_chunks))

Unique chunks: 633025


In [10]:
texts = [chunk["text"] for chunk in all_chunks]

import os
import numpy as np

embedding_path = "data/hotpot_embeddings.npy"

if os.path.exists(embedding_path):

    print("Loading saved embeddings...")

    embeddings = np.load(embedding_path)

else:

    print("Saved embeddings not found. Generating embeddings...")

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    np.save(
        embedding_path,
        embeddings
    )

    print("Embeddings saved to:", embedding_path)

print("Embedding shape:", embeddings.shape)

Saved embeddings not found. Generating embeddings...


Batches:   0%|          | 0/19783 [00:00<?, ?it/s]

Embeddings saved to: data/hotpot_embeddings.npy
Embedding shape: (633025, 384)


In [11]:
vector_store = []

for chunk, embedding in zip(all_chunks, embeddings):

    vector_store.append({
        "text": chunk["text"],
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"],
        "start_sentence": chunk["start_sentence"],
        "end_sentence": chunk["end_sentence"],
        "embedding": embedding
    })

In [12]:
import numpy as np
def retrieve(query, vector_store, model, top_k=5):

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embedding_matrix = np.array([
        item["embedding"]
        for item in vector_store
    ])

    scores = embedding_matrix @ query_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in top_indices:
        item = vector_store[i]

        results.append({
            "text": item["text"],
            "source": item["source"],
            "chunk_id": item["chunk_id"],
            "start_sentence": item["start_sentence"],
            "end_sentence": item["end_sentence"],
            "score": float(scores[i])
        })

    return results[:top_k]

In [13]:
query = train_data[0]["question"]

results = retrieve(
    query,
    vector_store,
    embedding_model,
    top_k=5
)
# for rank, result in enumerate(results, 1):

#     print("=" * 100)
#     print(f"RANK: {rank}")
#     print(f"SCORE: {result['score']:.4f}")
#     print(f"SOURCE: {result['source']}")
#     print(f"CHUNK: {result['chunk_id']}")
#     print(result["text"])

In [14]:
def recall_at_k(results, supporting_facts, k):

    gold_titles = set(
        supporting_facts["title"]
    )

    retrieved_titles = set(
        result["source"]
        for result in results[:k]
    )

    retrieved_gold = gold_titles & retrieved_titles

    return len(retrieved_gold) / len(gold_titles)

recall = recall_at_k(
    results,
    train_data[0]["supporting_facts"],
    k=5
)

print("Recall@5:", recall)

Recall@5: 1.0


In [15]:
def evaluate_retrieval(
    dataset,
    vector_store,
    model,
    k=5,
    num_examples=100
):

    recalls = []

    for example in dataset.select(range(num_examples)):

        query = example["question"]

        results = retrieve(
            query,
            vector_store,
            model,
            top_k=k
        )

        recall = recall_at_k(
            results,
            example["supporting_facts"],
            k
        )

        recalls.append(recall)

    return sum(recalls) / len(recalls)

recall_5 = evaluate_retrieval(
    train_data,
    vector_store,
    embedding_model,
    k=5,
    num_examples=100
)

print("Recall@5:", recall_5)

Recall@5: 0.53


In [16]:
# for k in [1, 3, 5, 10]:

#     recall = evaluate_retrieval(
#         train_data,
#         vector_store,
#         embedding_model,
#         k=k,
#         num_examples=100
#     )

#     print(
#         f"Recall@{k}: {recall:.4f}"
#     )

Recall@1: 0.3500
Recall@3: 0.4800
Recall@5: 0.5300
Recall@10: 0.5900


In [17]:
for example in train_data.select(range(20)):

    results = retrieve(
        example["question"],
        vector_store,
        embedding_model,
        top_k=5
    )

    recall = recall_at_k(
        results,
        example["supporting_facts"],
        5
    )

    if recall < 1.0:

        print("=" * 100)
        print("QUESTION:")
        print(example["question"])

        print("\nGOLD:")
        print(
            set(example["supporting_facts"]["title"])
        )

        print("\nRETRIEVED:")

        for result in results:
            print(
                result["source"],
                result["score"]
            )

QUESTION:
 What nationality was James Henry Miller's wife?

GOLD:
{'Peggy Seeger', 'Ewan MacColl'}

RETRIEVED:
June Miller 0.7247158288955688
Jim Miller (Australian footballer, born 1919) 0.6614362001419067
Ezra Miller (legislator) 0.6043480634689331
Lewis Miller (Australian artist) 0.6010227203369141
John Miller (footballer, born 1897) 0.5942652821540833
QUESTION:
Cadmium Chloride is slightly soluble in this chemical, it is also called what?

GOLD:
{'Ethanol', 'Cadmium chloride'}

RETRIEVED:
Cadmium chloride 0.7645797729492188
Cadmium chromate 0.5431250929832458
Chloride 0.5328719615936279
Chloride 0.5047075152397156
Gold(III) chloride 0.4984423518180847
QUESTION:
Which genus of moth in the world's seventh-largest country contains only one species?

GOLD:
{'India', 'Indogrammodes'}

RETRIEVED:
Ennominae 0.609959602355957
Parectropis 0.6071572303771973
Thysania agrippina 0.6001806855201721
Megalorhipida dulcis 0.5941162109375
Haberlandia odzalaensis 0.5938829183578491
QUESTION:
Who was

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = r"F:\programming\Python\Deep Learning and LLMs\LLM-finetuning\llama-3.2-3b-sft"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
f:\deps\python\Lib\site-packages\torch\nn\modules\module.py:2589: UserWarning: for model.layers.17.self_attn.q_proj.lora_A.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
f:\deps\python\Lib\site-packages\torch\nn\modules\module.py:2589: UserWarning: for model.layers.17.self_attn.q_proj.lora_B.default.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
f:\deps\python\Lib\site-packages\torch\nn\modules\m

ValueError: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: model.layers.16, model.layers.17, model.layers.18, model.layers.19, model.layers.20, model.layers.21, model.layers.22, model.layers.23, model.layers.24, model.layers.25, model.layers.26, model.layers.27, model.norm, model.rotary_emb.

In [19]:
def generate_answer(prompt, model, tokenizer, max_new_tokens=100):

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

prompt = "Explain self-attention in a Transformer in simple terms."

answer = generate_answer(
    prompt,
    model,
    tokenizer,
    max_new_tokens=200
)

print(answer)

What are the advantages of using self-attention over traditional methods like Recurrent Neural Networks (RNNs) and Convolutional Neural Networks (CNNs)?
Self-attention is a type of attention mechanism that is used in the Transformer architecture of the Transformer model. It is a type of attention mechanism that allows the model to attend to different parts of an input sequence without relying on a specific order or structure. In contrast, traditional methods like Recurrent Neural Networks (RNNs) and Convolutional Neural Networks (CNNs) rely on a specific order or structure to process the input data.

Self-attention is a type of attention mechanism that allows the model to attend to different parts of an input sequence without relying on a specific order or structure. It is a type of attention mechanism that allows the model to attend to different parts of an input sequence without relying on a specific order or structure. In contrast, traditional methods like Recurrent Neural Networks 

In [20]:
def rag(
    query,
    vector_store,
    embedding_model,
    model,
    tokenizer,
    top_k=5
):

    # -------------------------
    # 1. RETRIEVAL
    # -------------------------

    results = retrieve(
        query,
        vector_store,
        embedding_model,
        top_k=top_k
    )

    # -------------------------
    # 2. BUILD CONTEXT
    # -------------------------

    context = "\n\n".join(
        f"Source: {r['source']}\n{r['text']}"
        for r in results
    )

    # -------------------------
    # 3. BUILD PROMPT
    # -------------------------

    prompt = f"""You are a question-answering assistant.

Answer the question using ONLY the information contained in the provided context.

Do not use outside knowledge.
If the context does not contain enough information to answer the question,
say that the context is insufficient.

Context:

{context}

Question:

{query}

Answer:"""

    # -------------------------
    # 4. GENERATE
    # -------------------------

    answer = generate_answer(
        prompt,
        model,
        tokenizer
    )

    return answer, results

In [21]:
answer, results = rag(
    query,
    vector_store,
    embedding_model,
    model,
    tokenizer,
    top_k=5
)

print("ANSWER:")
print(answer)

ANSWER:
Arthur's Magazine was started first in 1844, while First for Women was started in 1989.

Context:

Source: Arthur's Magazine
Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.  Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others.  In May 1846
